# Chat with Your Database
## Natural Language to SQL using Oracle AI Database 26ai — Select AI

**KrishAI Technologies | April 2026**

---

> **What we build:** A live retail analytics database where you type plain English questions
> and Oracle AI Database 26ai generates, executes, and explains the SQL — zero external infrastructure required.

---

### What You Will Learn

| Topic | Description |
|---|---|
| `oracledb` thin client | Connect Python to Oracle Autonomous Database — no Oracle Client install needed |
| Retail schema setup | Create 4 tables + load ~1,750 rows of synthetic data with Faker |
| `DBMS_CLOUD_AI` | Register an LLM provider (OpenAI or Cohere) inside Oracle |
| **SELECT AI** | Execute natural language queries directly in SQL |
| **SHOWSQL** | See the SQL Oracle generated from your English question |
| **NARRATE** | Get a plain-English business summary instead of a table |
| **CHAT** | Multi-turn conversational data exploration with context retention |

### Prerequisites

| Requirement | Status |
|---|---|
| Oracle Cloud Free Tier account | See `ORACLE_CLOUD_SETUP.md` Step 1 |
| Autonomous Database 26ai (ATP) provisioned | See `ORACLE_CLOUD_SETUP.md` Step 2 |
| Wallet zip extracted to `./wallet/` | See `ORACLE_CLOUD_SETUP.md` Step 3 |
| OpenAI API key or Cohere API key | See `ORACLE_CLOUD_SETUP.md` Step 6/7 |
| `.env` file filled in from `.env.example` | See `ORACLE_CLOUD_SETUP.md` Step 8 |
| `uv sync` run successfully | See `ORACLE_CLOUD_SETUP.md` Step 9 |

**Quick start (3 commands):**
```bash
uv sync
cp .env.example .env   # then fill in your real values
uv run jupyter notebook
```

### Architecture — How Select AI Works

```
┌──────────────────────────────────────────────────────────────────┐
│  This Notebook (Python)                                          │
│                                                                  │
│   cur.execute("SELECT AI top 5 products by revenue?")            │
└──────────────────────┬───────────────────────────────────────────┘
                       │  python-oracledb + TLS wallet
                       ▼
┌──────────────────────────────────────────────────────────────────┐
│  Oracle Autonomous Database 26ai (Always Free Tier)              │
│                                                                  │
│  DBMS_CLOUD_AI reads your schema → calls LLM → runs the SQL     │
│                                                                  │
│  ┌────────────────┐          ┌──────────────────────────────┐    │
│  │ Your Schema    │          │ LLM Provider                 │    │
│  │ CUSTOMERS      │◄─────────│ OpenAI GPT-4o                │    │
│  │ PRODUCTS       │  SQL     │ or Cohere Command R+         │    │
│  │ ORDERS         │  result  │ (your choice)                │    │
│  │ ORDER_ITEMS    │          └──────────────────────────────┘    │
│  └────────────────┘                                              │
└──────────────────────────────────────────────────────────────────┘
```

**The key insight:** The LLM never sees your data — only your schema (table and column names).
The SQL runs inside Oracle, so all data governance and security controls remain in effect.

---
## Section 1: Environment Setup
*~5 minutes | Install packages, load configuration, connect to Oracle Autonomous Database 26ai*

---

In [ ]:
# Verify all packages are installed and import everything we need
import sys
import os
import json
import random
import importlib.util
from datetime import date, timedelta
from pathlib import Path
from IPython.display import display

# --- Package check: fail fast with a helpful message ---
required = {
    "oracledb":  "oracledb>=3.4.2",
    "faker":     "faker>=25.0.0",
    "dotenv":    "python-dotenv>=1.0.0",
    "pandas":    "pandas>=2.0.0",
    "rich":      "rich>=14.0.0",
    "tabulate":  "tabulate>=0.9.0",
}
missing = [pkg for mod, pkg in required.items() if not importlib.util.find_spec(mod)]
if missing:
    raise SystemExit(
        f"Missing packages: {missing}\n"
        "Fix: run  uv sync  in your terminal and restart the kernel."
    )

# --- All imports ---
import oracledb
import pandas as pd
from dotenv import load_dotenv
from faker import Faker
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()
fake    = Faker()
Faker.seed(42)
random.seed(42)

console.print(Panel(
    f"[bold green]All packages verified[/bold green]\n"
    f"oracledb {oracledb.__version__}  |  "
    f"pandas {pd.__version__}  |  "
    f"Python {sys.version.split()[0]}",
    title="Environment"
))

╭────────────────────────────────────────────────── Environment ──────────────────────────────────────────────────╮
│ All packages verified                                                                                           │
│ oracledb 3.4.2  |  pandas 3.0.2  |  Python 3.12.9                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Load .env and validate all required variables exist on disk
load_dotenv()

REQUIRED_ENV = ["DB_USER", "DB_PASSWORD", "DB_DSN", "WALLET_DIR", "WALLET_PASSWORD"]
missing_env = [v for v in REQUIRED_ENV if not os.getenv(v)]
if missing_env:
    raise EnvironmentError(
        f"Missing environment variables: {missing_env}\n"
        "Copy .env.example → .env and fill in your values."
    )

wallet_dir = Path(os.getenv("WALLET_DIR"))
if not wallet_dir.exists():
    raise FileNotFoundError(
        f"Wallet directory not found: {wallet_dir}\n"
        "Extract your wallet zip into that directory. See ORACLE_CLOUD_SETUP.md Step 3."
    )
if not (wallet_dir / "tnsnames.ora").exists():
    raise FileNotFoundError(
        f"tnsnames.ora not found in {wallet_dir}\n"
        "Extract the full wallet zip — not just the certificate files."
    )

console.print("[bold green]✓[/bold green] Environment loaded")
console.print(f"  DB_USER  = {os.getenv('DB_USER')}")
console.print(f"  DB_DSN   = {os.getenv('DB_DSN')}")
console.print(f"  WALLET   = {wallet_dir}")

✓ Environment loaded

DB_USER  = ADMIN

DB_DSN   = oraclekrish_high

WALLET   = /Users/sourangshupal/Downloads/oracle-yt/Wallet_ORACLEKRISH

In [ ]:
# Connect to Oracle Autonomous Database 26ai
# oracledb uses thin mode by default — no Oracle Client libraries required
try:
    connection = oracledb.connect(
        user            = os.getenv("DB_USER"),
        password        = os.getenv("DB_PASSWORD"),
        dsn             = os.getenv("DB_DSN"),
        config_dir      = str(wallet_dir),
        wallet_location = str(wallet_dir),
        wallet_password = os.getenv("WALLET_PASSWORD"),
    )
except oracledb.DatabaseError as e:
    code = getattr(e.args[0], "code", None)
    if code == 12154:
        raise ConnectionError(
            f"ORA-12154: DB_DSN='{os.getenv('DB_DSN')}' not found in tnsnames.ora\n"
            f"Check: grep -i '{os.getenv('DB_DSN')}' {wallet_dir}/tnsnames.ora"
        ) from e
    elif code == 1017:
        raise ConnectionError(
            "ORA-01017: Invalid username/password — verify DB_PASSWORD in your .env file"
        ) from e
    raise

# Verify Select AI is available (ADB-only feature)
with connection.cursor() as cur:
    cur.execute(
        "SELECT object_name FROM all_objects "
        "WHERE object_name = 'DBMS_CLOUD_AI' AND object_type = 'PACKAGE'"
    )
    has_select_ai = cur.fetchone() is not None

if not has_select_ai:
    raise RuntimeError(
        "DBMS_CLOUD_AI package not found.\n"
        "Select AI is only available on Oracle Autonomous Database (ATP/ADW).\n"
        "It does NOT work on Oracle XE, Oracle Free Edition, or on-premises installations."
    )

db_version = connection.version  # 26ai reports as 23.26.x internally
console.print(Panel(
    f"[bold green]Connected to Oracle AI Database 26ai[/bold green]\n"
    f"Version        : {db_version}\n"
    f"User           : {connection.username}\n"
    f"DBMS_CLOUD_AI  : [green]Available[/green] — Select AI ready",
    title="Oracle Connection",
    border_style="green"
))

╭─────────────────────────────────────────────── Oracle Connection ───────────────────────────────────────────────╮
│ Connected to Oracle AI Database 26ai                                                                            │
│ Version        : 23.26.2.1.0                                                                                    │
│ User           : ADMIN                                                                                          │
│ DBMS_CLOUD_AI  : Available — Select AI ready                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Section 2: Schema & Data Load
* Create 4 tables and load ~1,750 rows of realistic synthetic data *

We build a **retail sales schema** — universally understood, rich enough to produce interesting SQL,
and requiring no domain expertise.

| Table | Rows | Purpose in Demo |
|---|---|---|
| `CUSTOMERS` | 200 | Filter by city, age, segment (Premium/Standard/Budget) |
| `PRODUCTS` | 50 | Category analysis, pricing queries |
| `ORDERS` | 500 | Time-series (Jan 2025 – Apr 2026), status breakdowns |
| `ORDER_ITEMS` | ~1,500 | Revenue calculations, top products, discounts |

> **Idempotent:** Run this section multiple times safely — it drops and recreates the tables each time.

---

In [ ]:
# Create the 4-table retail sales schema
# Oracle syntax: GENERATED ALWAYS AS IDENTITY for auto-increment primary keys

DDL = {
    "CUSTOMERS": """
        CREATE TABLE customers (
            id       NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            name     VARCHAR2(100) NOT NULL,
            email    VARCHAR2(150) NOT NULL UNIQUE,
            city     VARCHAR2(80)  NOT NULL,
            age      NUMBER(3)     NOT NULL,
            segment  VARCHAR2(20)  NOT NULL
        )
    """,
    "PRODUCTS": """
        CREATE TABLE products (
            id             NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            name           VARCHAR2(150) NOT NULL,
            category       VARCHAR2(60)  NOT NULL,
            unit_price     NUMBER(10,2)  NOT NULL,
            stock_quantity NUMBER(8)     NOT NULL
        )
    """,
    "ORDERS": """
        CREATE TABLE orders (
            id            NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            customer_id   NUMBER        NOT NULL REFERENCES customers(id),
            order_date    DATE          NOT NULL,
            status        VARCHAR2(20)  NOT NULL,
            total_amount  NUMBER(12,2)  NOT NULL
        )
    """,
    "ORDER_ITEMS": """
        CREATE TABLE order_items (
            id          NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            order_id    NUMBER        NOT NULL REFERENCES orders(id),
            product_id  NUMBER        NOT NULL REFERENCES products(id),
            quantity    NUMBER(5)     NOT NULL,
            unit_price  NUMBER(10,2)  NOT NULL,
            discount    NUMBER(5,2)   DEFAULT 0
        )
    """,
}

def drop_table_if_exists(cur, table_name):
    """Drop table silently if it doesn't exist (ORA-00942)."""
    try:
        cur.execute(f"DROP TABLE {table_name} CASCADE CONSTRAINTS")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 942:  # 942 = table or view does not exist
            raise

with connection.cursor() as cur:
    # Drop in reverse FK order to avoid constraint violations
    for t in ["ORDER_ITEMS", "ORDERS", "PRODUCTS", "CUSTOMERS"]:
        drop_table_if_exists(cur, t)
    # Create in FK order
    for name, ddl in DDL.items():
        cur.execute(ddl)
        console.print(f"  [green]Created[/green] {name}")
    connection.commit()

console.print("[bold green]✓[/bold green] Schema created")

Created CUSTOMERS

Created PRODUCTS

Created ORDERS

Created ORDER_ITEMS

✓ Schema created

In [ ]:
# Generate and insert 200 customers + 50 products using Faker

# 10 cities → meaningful results for "sales by region" queries
CITIES = [
    "New York", "Los Angeles", "Chicago", "Houston", "Phoenix",
    "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"
]

# Realistic segment distribution: 20% Premium, 50% Standard, 30% Budget
SEGMENTS = ["Premium"] * 20 + ["Standard"] * 50 + ["Budget"] * 30

# 5 product categories, 10 products each = 50 products total
# Price ranges per category reflect realistic retail pricing
CATEGORIES = {
    "Electronics":   (80,  500),
    "Clothing":      (15,  120),
    "Home & Garden": (10,  250),
    "Sports":        (20,  180),
    "Books":         (5,   60),
}

customer_rows = [
    (
        fake.name(),
        fake.unique.email(),
        random.choice(CITIES),
        random.randint(18, 75),
        random.choice(SEGMENTS),
    )
    for _ in range(200)
]

product_rows = [
    (
        fake.bs().title()[:150],       # product name from business buzzwords
        category,
        round(random.uniform(lo, hi), 2),
        random.randint(0, 500),        # stock quantity
    )
    for category, (lo, hi) in CATEGORIES.items()
    for _ in range(10)
]

with connection.cursor() as cur:
    cur.executemany(
        "INSERT INTO customers(name, email, city, age, segment) VALUES(:1,:2,:3,:4,:5)",
        customer_rows
    )
    cur.executemany(
        "INSERT INTO products(name, category, unit_price, stock_quantity) VALUES(:1,:2,:3,:4)",
        product_rows
    )
    connection.commit()

console.print(
    f"[bold green]✓[/bold green] Inserted "
    f"{len(customer_rows)} customers across {len(CITIES)} cities, "
    f"{len(product_rows)} products across {len(CATEGORIES)} categories"
)

✓ Inserted 200 customers across 10 cities, 50 products across 5 categories

In [ ]:
# Generate and insert 500 orders + ~1,500 order_items
#
# Date range: Jan 2025 – Apr 2026 (16 months)
# This ensures Q1 2026 (Jan–Mar) and "last 90 days" queries return real data
#
# Status distribution: 70% completed, 20% pending, 10% cancelled
# Reflects a healthy but imperfect pipeline for interesting narrative queries

START_DATE = date(2025, 1, 1)
END_DATE   = date(2026, 4, 28)
STATUSES   = ["completed"] * 70 + ["pending"] * 20 + ["cancelled"] * 10

# Fetch customer and product IDs we just inserted
with connection.cursor() as cur:
    cur.execute("SELECT id FROM customers ORDER BY id")
    cust_ids = [r[0] for r in cur.fetchall()]

    cur.execute("SELECT id, unit_price FROM products ORDER BY id")
    products_map = {r[0]: float(r[1]) for r in cur.fetchall()}
    prod_ids = list(products_map.keys())

# Build orders and their items together (so total_amount is accurate)
all_item_rows = []

with connection.cursor() as cur:
    id_var = cur.var(oracledb.NUMBER)   # reusable variable for RETURNING INTO

    for _ in range(500):
        cust_id    = random.choice(cust_ids)
        order_date = START_DATE + timedelta(
            days=random.randint(0, (END_DATE - START_DATE).days)
        )
        status     = random.choice(STATUSES)
        n_items    = random.randint(1, 5)

        # Build line items and compute order total
        line_items = []
        total_amount = 0.0
        for _ in range(n_items):
            pid      = random.choice(prod_ids)
            qty      = random.randint(1, 8)
            up       = products_map[pid]
            discount = random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20])
            total_amount += up * qty * (1 - discount)
            line_items.append((pid, qty, up, discount))

        # Insert order and capture generated ID via RETURNING INTO
        cur.execute(
            "INSERT INTO orders(customer_id, order_date, status, total_amount) "
            "VALUES(:1, :2, :3, :4) RETURNING id INTO :5",
            [cust_id, order_date, status, round(total_amount, 2), id_var]
        )
        new_order_id = id_var.getvalue()[0]

        for pid, qty, up, disc in line_items:
            all_item_rows.append((new_order_id, pid, qty, up, disc))

    # Bulk insert all order items
    cur.executemany(
        "INSERT INTO order_items(order_id, product_id, quantity, unit_price, discount) "
        "VALUES(:1, :2, :3, :4, :5)",
        all_item_rows
    )
    connection.commit()

console.print(
    f"[bold green]✓[/bold green] Inserted 500 orders and {len(all_item_rows)} order items"
)

✓ Inserted 500 orders and 1561 order items

In [ ]:
# Verify row counts and check date coverage

with connection.cursor() as cur:
    counts = {}
    for t in ["CUSTOMERS", "PRODUCTS", "ORDERS", "ORDER_ITEMS"]:
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        counts[t] = cur.fetchone()[0]

    cur.execute(
        "SELECT MIN(order_date), MAX(order_date) FROM orders WHERE status = 'completed'"
    )
    min_date, max_date = cur.fetchone()

    # Quick preview: top cities by customer count
    cur.execute(
        "SELECT city, COUNT(*) AS customers "
        "FROM customers GROUP BY city ORDER BY customers DESC"
    )
    city_rows = cur.fetchall()

# Summary table
tbl = Table(title="Schema Loaded Successfully", show_header=True, header_style="bold cyan")
tbl.add_column("Table", style="bold")
tbl.add_column("Row Count", justify="right")
for table_name, count in counts.items():
    tbl.add_row(table_name, str(count))
console.print(tbl)

console.print(
    f"\n  Order date range (completed): "
    f"{min_date.date()} → {max_date.date()}"
)

# Show city distribution as a DataFrame
df_cities = pd.DataFrame(city_rows, columns=["City", "Customers"])
print("\nCustomer distribution by city:")
display(df_cities)

Schema Loaded Successfully 
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Table       ┃ Row Count ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ CUSTOMERS   │       200 │
│ PRODUCTS    │        50 │
│ ORDERS      │       500 │
│ ORDER_ITEMS │      1561 │
└─────────────┴───────────┘

Order date range (completed): 2025-01-05 → 2026-04-25


Customer distribution by city:


,City,Customers
0,Los Angeles,30
1,Dallas,25
2,Houston,21
3,New York,20
4,Chicago,19
5,Phoenix,18
6,San Diego,18
7,Philadelphia,18
8,San Antonio,17
9,San Jose,14


---
## Section 3: Select AI Profile Setup
* Register an LLM provider inside Oracle via DBMS_CLOUD_AI *

Setting up Select AI requires two steps:

**Step A — Credential:** Store your LLM API key securely inside Oracle using `DBMS_CLOUD.CREATE_CREDENTIAL`.  
The key is encrypted — you can never retrieve it after storage.

**Step B — Profile:** Register the LLM provider, model, and the tables you want to expose using `DBMS_CLOUD_AI.CREATE_PROFILE`.  
The profile acts as an access-controlled contract — Select AI can only query tables you explicitly list.

> **Session-scoped:** The active profile must be set with `DBMS_CLOUD_AI.SET_PROFILE` each time you open a new database connection.
> If you restart the kernel, re-run this section before the demo cells.

> **LLM provider flexibility:** Oracle 26ai supports OpenAI, Cohere, Azure OpenAI, and others.
> The notebook auto-detects which provider to use based on which API key you set in `.env`.

---

In [ ]:
# Detect LLM provider and create credential

OPENAI_KEY = os.getenv("OPENAI_API_KEY", "")
COHERE_KEY = os.getenv("COHERE_API_KEY", "")

if not OPENAI_KEY and not COHERE_KEY:
    raise EnvironmentError(
        "No LLM provider key found in .env\n"
        "Set either OPENAI_API_KEY or COHERE_API_KEY."
    )

USE_OPENAI   = bool(OPENAI_KEY)
PROVIDER     = "openai"          if USE_OPENAI else "cohere"
CRED_NAME    = "OPENAI_CRED"     if USE_OPENAI else "COHERE_CRED"
CRED_USER    = "OPENAI"          if USE_OPENAI else "COHERE"
CRED_PASS    = OPENAI_KEY        if USE_OPENAI else COHERE_KEY
MODEL        = "gpt-4o"          if USE_OPENAI else "command-r-plus"
PROFILE_NAME = "OPENAI_PROFILE"  if USE_OPENAI else "COHERE_PROFILE"

def drop_credential_if_exists(cur, name):
    """Drop credential silently if it doesn't exist."""
    try:
        cur.execute("BEGIN DBMS_CLOUD.DROP_CREDENTIAL(:1); END;", [name])
    except oracledb.DatabaseError as e:
        msg = str(e).lower()
        if "ora-20404" not in msg and "does not exist" not in msg:
            raise

with connection.cursor() as cur:
    drop_credential_if_exists(cur, CRED_NAME)
    cur.execute(
        """
        BEGIN
          DBMS_CLOUD.CREATE_CREDENTIAL(
            credential_name => :1,
            username        => :2,
            password        => :3
          );
        END;
        """,
        [CRED_NAME, CRED_USER, CRED_PASS]
    )
    connection.commit()

console.print(
    f"[bold green]✓[/bold green] Credential '{CRED_NAME}' created\n"
    f"  Provider : {PROVIDER}\n"
    f"  Model    : {MODEL}"
)

✓ Credential 'OPENAI_CRED' created
  Provider : openai
  Model    : gpt-4o

In [ ]:
# Create the DBMS_CLOUD_AI profile
# The object_list controls exactly which tables the AI can query — security by design

DB_OWNER = os.getenv("DB_USER", "ADMIN").upper()  # Oracle schema names are UPPERCASE

profile_attributes = json.dumps({
    "provider":        PROVIDER,
    "credential_name": CRED_NAME,
    "object_list": [
        {"owner": DB_OWNER, "name": "CUSTOMERS"},
        {"owner": DB_OWNER, "name": "PRODUCTS"},
        {"owner": DB_OWNER, "name": "ORDERS"},
        {"owner": DB_OWNER, "name": "ORDER_ITEMS"},
    ],
    "model": MODEL,
})

def drop_profile_if_exists(cur, name):
    """Drop AI profile silently if it doesn't exist."""
    try:
        cur.execute("BEGIN DBMS_CLOUD_AI.DROP_PROFILE(:1); END;", [name])
    except oracledb.DatabaseError as e:
        msg = str(e).lower()
        if "does not exist" not in msg and "ora-20404" not in msg:
            raise

with connection.cursor() as cur:
    drop_profile_if_exists(cur, PROFILE_NAME)
    cur.execute(
        """
        BEGIN
          DBMS_CLOUD_AI.CREATE_PROFILE(
            profile_name => :1,
            attributes   => :2
          );
        END;
        """,
        [PROFILE_NAME, profile_attributes]
    )
    connection.commit()

console.print(
    f"[bold green]✓[/bold green] Profile '{PROFILE_NAME}' created\n"
    f"  Tables exposed : CUSTOMERS, PRODUCTS, ORDERS, ORDER_ITEMS\n"
    f"  Schema owner   : {DB_OWNER}\n"
    f"  LLM provider   : {PROVIDER} ({MODEL})"
)

✓ Profile 'OPENAI_PROFILE' created
  Tables exposed : CUSTOMERS, PRODUCTS, ORDERS, ORDER_ITEMS
  Schema owner   : ADMIN
  LLM provider   : openai (gpt-4o)

In [ ]:
# Activate the profile for this session
#
# SET_PROFILE is SESSION-SCOPED. If you restart the kernel or reconnect,
# re-run this cell before running any SELECT AI queries.

with connection.cursor() as cur:
    cur.execute(
        "BEGIN DBMS_CLOUD_AI.SET_PROFILE(profile_name => :1); END;",
        [PROFILE_NAME]
    )

console.print(Panel(
    f"[bold green]Select AI is READY![/bold green]\n\n"
    f"Profile  : {PROFILE_NAME}\n"
    f"Provider : {PROVIDER}\n"
    f"Model    : {MODEL}\n\n"
    f"[italic dim]Type English → Oracle generates SQL → Results returned[/italic dim]",
    title="Select AI Active",
    border_style="green"
))

╭─────────────────────────────────────────────── Select AI Active ────────────────────────────────────────────────╮
│ Select AI is READY!                                                                                             │
│                                                                                                                 │
│ Profile  : OPENAI_PROFILE                                                                                       │
│ Provider : openai                                                                                               │
│ Model    : gpt-4o                                                                                               │
│                                                                                                                 │
│ Type English → Oracle generates SQL → Results returned                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Section 4: Live Demo — The Four Select AI Modes
*~8 minutes | Type English, watch Oracle do the work*

Oracle AI Database 26ai provides **four distinct modes** for natural language interaction:

| Mode | Syntax | What it Returns |
|---|---|---|
| **SELECT AI** | `SELECT AI <question>` | Executes the generated SQL, returns a data table |
| **SHOWSQL** | `SELECT AI SHOWSQL <question>` | Shows the generated SQL without executing it |
| **NARRATE** | `SELECT AI NARRATE <question>` | Returns a plain-English business summary |
| **CHAT** | `SELECT AI CHAT <question>` | Free-form conversational mode with context retention |

> **Key insight:** The LLM generates the SQL based on your schema metadata.
> SHOWSQL is the most educational mode — it reveals Oracle's "thinking". 
> Ask the same question in SHOWSQL that you asked in SELECT AI to see the exact query behind the answer.

---

In [ ]:
# Helper function

def run_select_ai(nl_question: str, mode: str = "") -> "pd.DataFrame | str":
    """
    Execute a Select AI query against the Oracle 26ai database.

    Parameters
    ----------
    nl_question : str
        Plain English question to ask the database.
    mode : str
        One of: '' (execute), 'SHOWSQL', 'NARRATE', 'CHAT'

    Returns
    -------
    pd.DataFrame for execute mode, str for all other modes.
    """
    mode_str = f" {mode}" if mode else ""
    sql = f"SELECT AI{mode_str} {nl_question}"

    console.print(Panel(
        f"[bold yellow]Question:[/bold yellow] {nl_question}",
        title=f"Select AI — {mode if mode else 'EXECUTE'}",
        border_style="yellow"
    ))

    with connection.cursor() as cur:
        cur.execute(sql)
        columns = [col[0] for col in (cur.description or [])]
        rows    = cur.fetchall()

    if mode in ("SHOWSQL", "NARRATE", "CHAT"):
        # These modes return a single text/LOB column
        raw    = rows[0][0] if rows else "(no response)"
        result = raw.read() if hasattr(raw, "read") else str(raw)
        console.print(Panel(result, title="Response", border_style="green"))
        return result
    else:
        if not rows:
            console.print("[yellow]No results returned.[/yellow]")
            return pd.DataFrame(columns=columns)
        df = pd.DataFrame(rows, columns=columns)
        display(df)
        return df

### Mode 1: SELECT AI — Execute Natural Language Queries

The core demo. Type a question in plain English — Oracle generates the SQL and returns the results directly.
No SQL knowledge required.

In [ ]:
# SELECT AI — three live queries

# Query 1: Revenue analysis — tests JOIN across 3 tables + aggregation + sorting
df1 = run_select_ai("what were the top 5 products by total revenue from completed orders?")

# Query 2: Customer activity — tests date arithmetic + HAVING clause
df2 = run_select_ai("which customers have not placed any order in the last 90 days?")

# Query 3: Regional breakdown — tests GROUP BY + date range filtering
df3 = run_select_ai("show me total sales amount grouped by city for completed orders placed in 2026")

╭────────────────────────────────────────────── Select AI — EXECUTE ──────────────────────────────────────────────╮
│ Question: what were the top 5 products by total revenue from completed orders?                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,Product Name,Total Revenue
0,Re-Intermediate Front-End Systems,59321.55
1,Enable Enterprise Eyeballs,48362.44
2,Exploit Frictionless Systems,32460.12
3,Exploit Collaborative Solutions,26848.05
4,Integrate Transparent Relationships,24903.51


╭────────────────────────────────────────────── Select AI — EXECUTE ──────────────────────────────────────────────╮
│ Question: which customers have not placed any order in the last 90 days?                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,Customer Name,Email,City
0,Dennis Moody,ronaldstephens@example.net,New York
1,Tammy Gonzales,sbarker@example.com,San Diego
2,Kathryn Snyder,georgepamela@example.net,Dallas
3,David Davis,keyemily@example.com,San Diego
4,Pamela Thompson,lmoon@example.net,Phoenix
...,...,...,...
127,John Peterson,kingmichelle@example.org,San Diego
128,Jennifer Jones,hernandezlisa@example.com,San Diego
129,Joseph Hayes,levans@example.com,Dallas
130,Marc Lynch,mezajared@example.org,Los Angeles


╭────────────────────────────────────────────── Select AI — EXECUTE ──────────────────────────────────────────────╮
│ Question: show me total sales amount grouped by city for completed orders placed in 2026                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,City,Total Sales Amount
0,Los Angeles,20450.14
1,Phoenix,18927.24
2,Philadelphia,2234.14
3,Dallas,13058.52
4,Chicago,10089.86
5,San Jose,7169.54
6,Houston,19381.89
7,San Diego,1405.23
8,San Antonio,13692.96
9,New York,25268.17


### Mode 2: SHOWSQL — See the Generated SQL

SHOWSQL reveals the SQL that Oracle generated from your English question — **without executing it**.
This is the most educational part of the demo: you can see exactly how the AI interprets your question.

Great for:
- Teaching SQL to beginners using their own questions as examples
- Verifying the AI understood your intent before running on large datasets
- Debugging unexpected query results

In [ ]:
# SHOWSQL — reveal the SQL behind the same three questions
# Compare these with the results from Mode 1 above

sql1 = run_select_ai(
    "what were the top 5 products by total revenue from completed orders?",
    "SHOWSQL"
)

sql2 = run_select_ai(
    "which customers have not placed any order in the last 90 days?",
    "SHOWSQL"
)

sql3 = run_select_ai(
    "show me total sales amount grouped by city for completed orders placed in 2026",
    "SHOWSQL"
)

╭────────────────────────────────────────────── Select AI — SHOWSQL ──────────────────────────────────────────────╮
│ Question: what were the top 5 products by total revenue from completed orders?                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ SELECT                                                                                                          │
│     p."NAME" AS "Product Name",                                                                                 │
│     SUM(oi."UNIT_PRICE" * oi."QUANTITY" - oi."DISCOUNT") AS "Total Revenue"                                     │
│ FROM                                                                                                            │
│     "ADMIN"."ORDER_ITEMS" oi                                                                                    │
│ JOIN                                                                                                            │
│     "ADMIN"."ORDERS" o ON oi."ORDER_ID" = o."ID"                                                                │
│ JOIN                                                                                                            │
│     "ADMIN"."PRODUCTS" p ON oi."PRODUCT_ID" = p."ID"                                                            │
│ WHERE                                                                                                           │
│     UPPER(o."STATUS") = 'COMPLETED'                                                                             │
│ GROUP BY                                                                                                        │
│     p."NAME"                                                                                                    │
│ ORDER BY                                                                                                        │
│     "Total Revenue" DESC                                                                                        │
│ FETCH FIRST 5 ROWS ONLY                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Select AI — SHOWSQL ──────────────────────────────────────────────╮
│ Question: which customers have not placed any order in the last 90 days?                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ SELECT                                                                                                          │
│     c."NAME" AS "Customer Name",                                                                                │
│     c."EMAIL" AS "Email",                                                                                       │
│     c."CITY" AS "City"                                                                                          │
│ FROM                                                                                                            │
│     "ADMIN"."CUSTOMERS" c                                                                                       │
│ WHERE                                                                                                           │
│     c."ID" NOT IN (                                                                                             │
│         SELECT                                                                                                  │
│             o."CUSTOMER_ID"                                                                                     │
│         FROM                                                                                                    │
│             "ADMIN"."ORDERS" o                                                                                  │
│         WHERE                                                                                                   │
│             o."ORDER_DATE" >= SYSDATE - 90                                                                      │
│     )                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Select AI — SHOWSQL ──────────────────────────────────────────────╮
│ Question: show me total sales amount grouped by city for completed orders placed in 2026                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ SELECT                                                                                                          │
│     c."CITY" AS "City",                                                                                         │
│     SUM(o."TOTAL_AMOUNT") AS "Total_Sales_Amount"                                                               │
│ FROM                                                                                                            │
│     "ADMIN"."ORDERS" o                                                                                          │
│ JOIN                                                                                                            │
│     "ADMIN"."CUSTOMERS" c ON o."CUSTOMER_ID" = c."ID"                                                           │
│ WHERE                                                                                                           │
│     UPPER(o."STATUS") = 'COMPLETED'                                                                             │
│     AND EXTRACT(YEAR FROM o."ORDER_DATE") = 2026                                                                │
│ GROUP BY                                                                                                        │
│     c."CITY"                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Mode 3: NARRATE — Business Insights as Plain English

Instead of returning a table, NARRATE returns a human-readable paragraph summarising the data.

This demonstrates that Oracle 26ai is not just a query engine — it understands business context
and can communicate results directly to non-technical stakeholders such as executives and managers.

In [ ]:
# NARRATE — three business-summary queries

# Summary 1: Category performance for Q1 reporting
n1 = run_select_ai(
    "summarize total sales by product category for Q1 2026 in a business-friendly way",
    "NARRATE"
)

# Summary 2: Customer segment revenue analysis
n2 = run_select_ai(
    "which customer segments are driving the most revenue and what does that tell us about our customer base?",
    "NARRATE"
)

# Summary 3: Order pipeline health check
n3 = run_select_ai(
    "describe the overall health of our order pipeline — pending, completed, and cancelled orders",
    "NARRATE"
)

╭────────────────────────────────────────────── Select AI — NARRATE ──────────────────────────────────────────────╮
│ Question: summarize total sales by product category for Q1 2026 in a business-friendly way                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ In the first quarter of 2026, the total sales for each product category were as follows:                        │
│                                                                                                                 │
│ - Electronics: $74,937.11                                                                                       │
│ - Books: $8,504.70                                                                                              │
│ - Clothing: $13,913.58                                                                                          │
│ - Home & Garden: $47,557.81                                                                                     │
│ - Sports: $28,032.48                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Select AI — NARRATE ──────────────────────────────────────────────╮
│ Question: which customer segments are driving the most revenue and what does that tell us about our customer    │
│ base?                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ - The "Standard" customer segment is generating the highest revenue, indicating that this group is the most     │
│ significant contributor to sales.                                                                               │
│ - The "Budget" segment follows, showing a substantial contribution but less than the "Standard" segment.        │
│ - The "Premium" segment contributes the least to the revenue, suggesting that fewer high-end purchases are      │
│ being made compared to the other segments.                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Select AI — NARRATE ──────────────────────────────────────────────╮
│ Question: describe the overall health of our order pipeline — pending, completed, and cancelled orders          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

### Mode 4: CHAT — Conversational Data Exploration

CHAT enables free-form, multi-turn conversation grounded in your live database.
Users can ask follow-up questions, request comparisons, or drill down — all governed by Oracle's security controls.

Notice the third question uses **"those customers"** — Oracle retains context from the previous turn
and understands this refers to the premium segment customers from the previous answer.

This previews the **agentic AI direction** of Oracle AI Database 26ai.

In [ ]:
# CHAT — three-turn conversation demonstrating context retention

# Turn 1: Broad opening question
c1 = run_select_ai(
    "what can you tell me about our customer base?",
    "CHAT"
)

# Turn 2: Drill down on a specific dimension
c2 = run_select_ai(
    "which city has the highest number of premium segment customers?",
    "CHAT"
)

# Turn 3: Follow-up using context from Turn 2
# "those customers" refers back to the premium customers in the city identified in Turn 2
c3 = run_select_ai(
    "and what is the average order value for those customers?",
    "CHAT"
)

╭─────────────────────────────────────────────── Select AI — CHAT ────────────────────────────────────────────────╮
│ Question: what can you tell me about our customer base?                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ To provide insights about your customer base, I would need more specific information about your business, such  │
│ as the industry you operate in, the products or services you offer, and any available data on your customers.   │
│ However, I can offer some general areas to consider when analyzing a customer base:                             │
│                                                                                                                 │
│ 1. **Demographics**: Understand the age, gender, income level, education, and geographic location of your       │
│ customers. This helps in tailoring marketing strategies and product offerings.                                  │
│                                                                                                                 │
│ 2. **Psychographics**: Analyze the lifestyle, values, interests, and opinions of your customers. This can       │
│ provide deeper insights into their purchasing motivations.                                                      │
│                                                                                                                 │
│ 3. **Behavioral Data**: Look at purchasing patterns, frequency of purchases, average transaction value, and     │
│ customer loyalty. This helps in identifying your most valuable customers and understanding their buying habits. │
│                                                                                                                 │
│ 4. **Customer Segmentation**: Divide your customer base into distinct groups based on shared characteristics.   │
│ This allows for more targeted marketing and personalized customer experiences.                                  │
│                                                                                                                 │
│ 5. **Customer Feedback**: Gather and analyze feedback from customer reviews, surveys, and social media. This    │
│ can highlight areas for improvement and opportunities for new products or services.                             │
│                                                                                                                 │
│ 6. **Customer Lifetime Value (CLV)**: Calculate the total worth of a customer over the entire duration of their │
│ relationship with your business. This helps in prioritizing resources towards retaining high-value customers.   │
│                                                                                                                 │
│ 7. **Churn Rate**: Measure the rate at which customers stop doing business with you. Understanding why          │
│ customers leave can help in developing strategies to improve retention.                                         │
│                                                                                                                 │
│ 8. **Market Trends**: Stay informed about industry trends and how they might affect your customer base. This    │
│ can help in anticipating changes in customer needs and preferences.                                             │
│                                                                                                                 │
│ If you have specific data or questions about your customer base, feel free to share, and I can provide more     │
│ tailored insights.                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Select AI — CHAT ────────────────────────────────────────────────╮
│ Question: which city has the highest number of premium segment customers?                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ To accurately answer which city has the highest number of premium segment customers, specific data from a       │
│ particular industry or company is needed. This information can vary greatly depending on the context, such as   │
│ the type of service or product, the geographic region, and the time period in question. Typically, large        │
│ metropolitan areas with higher income levels, such as New York, London, or Tokyo, might have a higher number of │
│ premium segment customers due to their larger populations and economic status. However, without specific data,  │
│ it's impossible to definitively identify the city with the highest number of premium segment customers.         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Select AI — CHAT ────────────────────────────────────────────────╮
│ Question: and what is the average order value for those customers?                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Response ────────────────────────────────────────────────────╮
│ To calculate the average order value (AOV) for customers, you need to know the total revenue generated from     │
│ those customers and the total number of orders placed by them. The formula for calculating the average order    │
│ value is:                                                                                                       │
│                                                                                                                 │
│ [ \text{Average Order Value (AOV)} = \frac{\text{Total Revenue}}{\text{Total Number of Orders}} \]              │
│                                                                                                                 │
│ If you have the total revenue and the total number of orders, you can simply divide the revenue by the number   │
│ of orders to find the AOV. If you provide those numbers, I can help you calculate it.                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Section 5: Real-World Extensions
*~3 minutes | Where to take Select AI in production*

---

### 1. Oracle APEX AI App Generator

Oracle APEX (Application Express) is available in the same Always Free Tier.
With **APEX AI App Generator** (APEX 24.1+), you can generate a full web application
on top of this exact database schema in a single click — forms, reports, and charts — all driven by natural language.

**Access APEX:** From your ADB console → Database Actions → APEX

---

### 2. Agentic Workflows via PL/SQL

Oracle 26ai exposes `DBMS_CLOUD_AI.GENERATE` as a PL/SQL function, enabling
database-native agentic loops — query → reason → query → report — entirely inside stored procedures:

```sql
-- Run in SQL Developer Web (Database Actions → SQL)
DECLARE
  v_result CLOB;
BEGIN
  v_result := DBMS_CLOUD_AI.GENERATE(
    prompt       => 'Which products should we restock based on low inventory and high recent sales?',
    profile_name => 'OPENAI_PROFILE',
    action       => 'narrate'
  );
  DBMS_OUTPUT.PUT_LINE(v_result);
END;
```

---

### 3. Enterprise BI Integration

Connect **Power BI**, **Tableau**, or **Oracle Analytics Cloud** to the same Autonomous Database.
Business users query in plain English through those tools — Select AI translates behind the scenes.
No changes to the BI tool required.

---

### 4. Python RAG Pattern — Next Video Teaser

Combine Select AI with document context for grounded analytics:

In [ ]:
# This pattern is explored in depth in the next video: Oracle 26ai Vector Search

def rag_query(nl_question: str, doc_context: str) -> str:
    """
    Augment a Select AI query with external document context.
    Useful when your question requires knowledge not in the database schema.
    """
    # oracledb treats ANY ":word" in SQL as a bind variable placeholder.
    # Strip colons from all dynamic parts, then do a final safety pass on the full SQL.
    clean_context  = " ".join(doc_context.split())[:500].replace(":", " ")
    clean_question = nl_question.replace(":", " ")
    augmented      = f"Given this business context - {clean_context}. Question - {clean_question}"
    safe_sql       = f"SELECT AI NARRATE {augmented}".replace(":", " ")
    with connection.cursor() as cur:
        cur.execute(safe_sql)
        raw = cur.fetchone()[0]
        return raw.read() if hasattr(raw, "read") else str(raw)


# Example: inject a business rule from an external document
business_context = """
Our company launched a new loyalty program in Q1 2026. Premium customers who spent over 500 dollars
in the past 6 months qualify for a 15 percent discount on their next order.
"""

result = rag_query(
    "which of our premium customers qualify for the new loyalty discount",
    business_context
)
print(result)

The following premium customers qualify for the new loyalty discount because they spent over 500 dollars in the past 6 months:

1. Michael Stephens - bruce43@example.org - $739.99
2. Cristian Santos - lrobinson@example.com - $1128.69
3. Anna Gay - murraydavid@example.org - $3627.88
4. Dr. Jordan Hill PhD - jsanchez@example.org - $2202.4
5. Sophia Moore - craigjoseph@example.net - $529.59
6. John Peterson - kingmichelle@example.org - $5644.87
7. Brenda Levy - zcoffey@example.net - $5944.06
8. Shannon Ray - williamsjeremy@example.com - $2925.66
9. Jessica Meadows - vmerritt@example.com - $1979.53
10. Tiffany Vaughn - javierwashington@example.net - $1183
11. James Little - daniel37@example.org - $5463.55
12. Ashlee Jackson - rebecca05@example.org - $1936.22
13. Judith Maynard - lutzmelanie@example.com - $1373.45
14. John Boone - megan51@example.com - $3436.84
15. Madison Poole - ryangross@example.net - $6440.39
16. Larry Mason - hayeslisa@example.com - $2960.96
17. Kimberly Matthews - nic

---
## Wrap-up

You just built a **natural language analytics interface** on a real Oracle database — for free.

### What you accomplished in this notebook

| ✅ | Achievement |
|---|---|
| ✅ | Connected Python to Oracle AI Database 26ai (Always Free Tier) |
| ✅ | Loaded a production-quality retail schema with ~1,750 rows of synthetic data |
| ✅ | Registered an LLM provider (OpenAI GPT-4o or Cohere Command R+) inside Oracle |
| ✅ | Demonstrated **SELECT AI** — type English, get results |
| ✅ | Demonstrated **SHOWSQL** — see the AI-generated SQL transparently |
| ✅ | Demonstrated **NARRATE** — business summaries in plain English |
| ✅ | Demonstrated **CHAT** — multi-turn conversational queries with context retention |

### Try it yourself — free forever

**Oracle Cloud Always Free Tier:** https://cloud.oracle.com  
- Autonomous Database 26ai ATP: Always Free, 20 GB storage, forever  
- No credit card required to sign up  


---

*KrishAI Technologies | April 2026*  